# LLM Reasoning

**LLM Reasoning** (rozumowanie LLM) to zaawansowana funkcjonalność modeli językowych, która pozwala im na przeprowadzenie wewnętrznego procesu "myślenia" przed udzieleniem finalnej odpowiedzi. W przeciwieństwie do standardowego generowania tekstu, gdzie model od razu produkuje odpowiedź, reasoning pozwala modelowi na:

- **Rozważenie różnych podejść** do problemu przed wyborem najlepszego
- **Weryfikację własnych hipotez** i wykrycie potencjalnych błędów w rozumowaniu
- **Przeprowadzenie wieloetapowej analizy** złożonych problemów
- **Samokorektę** w trakcie procesu myślenia

Dzięki temu modele z reasoningiem osiągają znacznie lepsze wyniki w zadaniach wymagających:
- Rozwiązywania problemów matematycznych
- Analizy logicznej i dedukcji
- Programowania i debugowania kodu
- Planowania wieloetapowych działań
- Odpowiadania na pytania wymagające dogłębnej analizy

**Różnica między Chain-of-Thought a Reasoning:**

Chain-of-thought to technika promptowania, gdzie my instruujemy model, aby przedstawił swoje rozumowanie krok po kroku w odpowiedzi. Reasoning natomiast jest wbudowaną zdolnością modelu do wewnętrznego "myślenia" przed wygenerowaniem odpowiedzi - proces ten może być niewidoczny dla użytkownika lub dostępny jako osobny output.

**Modele z wbudowanym reasoningiem:**
- **o1-preview** i **o1-mini** - pierwsze modele OpenAI z natywnym reasoningiem
- **gpt-5.2** - najnowszy model z rozszerzonym reasoningiem (wprowadza najwyższy poziom `xhigh`)

**Ważne ograniczenia modeli reasoningowych:**

Modele z wbudowanym reasoningiem (o1, o3, gpt-5.x) **nie obsługują** następujących parametrów dostępnych w standardowych modelach:
- `temperature` - kontrola losowości odpowiedzi
- `top_p` - alternatywna kontrola próbkowania
- `presence_penalty` i `frequency_penalty` - kary za powtórzenia
- `logprobs`, `top_logprobs` - prawdopodobieństwa tokenów
- `logit_bias` - wpływ na wybór konkretnych tokenów

Parametry te są wyłączone, ponieważ modele reasoningowe używają wewnętrznych mechanizmów optymalizacji procesu myślenia, które są niezależne od standardowego próbkowania. Zamiast kontrolowania temperatury, kontrolujemy **głębokość rozumowania** poprzez parametr `reasoning_effort`.

W tym notebooku skupimy się na wykorzystaniu **gpt-5.2** i jego możliwości reasoningu.

[Dokumentacja OpenAI - Reasoning](https://platform.openai.com/docs/guides/reasoning)

## 1. Podstawowe użycie reasoningu z gpt-5.2

Model **gpt-5.2** obsługuje reasoning poprzez parametr `reasoning_effort`, który kontroluje jak intensywnie model "myśli" przed odpowiedzią. Możliwe wartości to:
- `"none"` - wyłączone rozumowanie, najszybsza odpowiedź (domyślne dla gpt-5.2)
- `"low"` - minimalne rozumowanie, szybka odpowiedź
- `"medium"` - zrównoważone podejście
- `"high"` - rozszerzone rozumowanie, dokładniejsza odpowiedź
- `"xhigh"` - maksymalne rozumowanie (nowość w gpt-5.2), najwyższa dokładność, może trwać 5-10 minut dla krytycznych decyzji

**Uwaga:** Wyższe poziomy reasoningu oznaczają większe zużycie tokenów i dłuższy czas odpowiedzi, ale zwiększają jakość rozwiązań złożonych problemów.

In [ ]:
from openai import OpenAI
import os
import textwrap

# Inicjalizacja klienta
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

In [ ]:
# Proste zadanie matematyczne
response = client.responses.create(
    model="gpt-5.2",
    input=[  # Response API używa input zamiast messages
        {"role": "system", "content": "Jesteś pomocnym asystentem matematycznym."},
        {"role": "user", "content": "Rozwiąż równanie: 2x + 5 = 13"}
    ],
    reasoning={
        "effort": "xhigh"  # włączenie reasoningu na poziomie najwyższym
    },
    max_output_tokens=2000  # limit tokenów odpowiedzi
)

# Wyświetlenie odpowiedzi
print(response.output_text)

# Wyświetlenie statystyk tokenów
print(f"\n=== STATYSTYKI TOKENÓW ===")
print(f"Input tokens:     {response.usage.input_tokens}")
print(f"Output tokens:    {response.usage.output_tokens}, w czym:")
print(f". Reasoning tokens: {response.usage.output_tokens_details.reasoning_tokens}")
print(f". Answer tokens:    {response.usage.output_tokens - response.usage.output_tokens_details.reasoning_tokens}")
print(f"Total tokens:     {response.usage.total_tokens}")

## 2. Porównanie: z reasoningiem vs. bez reasoningu

Zobaczmy różnicę między odpowiedzią z reasoningiem i bez reasoningu na tym samym, bardziej złożonym problemie.

In [ ]:
# Problem logiczny wymagający wieloetapowej analizy
# Reguła: wynik dla "a b c" to a * (a + b + c + 1)
#   Wynik: 3 3 3 : 30
problem = """
Przykłady:
1 2 3 : 7
4 1 1 : 28
2 2 2 : 14
3 2 1 : 21
1 1 1 : 4
2 1 1 : 10

Zadanie testowe:
3 3 3 : ?

Uzupełnij brakujące wartości w zadaniu testowym powyżej.

Format odpowiedzi:
Reguła: {krótki czystotekstowy wzór wyliczenia d, bez latexa ani formatowania}
Uzupełniony ciąg z zadania : {format: a b c : d}
"""

# Wersja BEZ reasoningu (standardowy gpt-5.2)
response_without = client.responses.create(
    model="gpt-5.2",
    input=[
        {"role": "system", "content": "Jesteś pomocnym asystentem."},
        {"role": "user", "content": problem}
    ],
    temperature=0,
    max_output_tokens=2000
    # BRAK parametru reasoning - model nie używa wewnętrznego procesu myślenia
)

print("=== ODPOWIEDŹ BEZ REASONINGU (gpt-5.2) ===")
print(response_without.output_text)
print()

In [ ]:
# Wersja Z reasoningiem (gpt-5.2 z wysokim poziomem reasoningu)
response_with = client.responses.create(
    model="gpt-5.2",
    input=[
        {"role": "system", "content": "Jesteś pomocnym asystentem."},
        {"role": "user", "content": problem}
    ],
    reasoning={
        "effort": "high"  # maksymalny poziom reasoningu dla dokładności
    },
    max_output_tokens=20000  # UWAGA: modele reasoningowe nie obsługują parametru temperature!
)

print("=== ODPOWIEDŹ Z REASONINGIEM (gpt-5.2) ===")
print(response_with.output_text)

# Porównanie zużycia tokenów
print(f"\n=== PORÓWNANIE ZUŻYCIA TOKENÓW ===")
print(f"BEZ reasoningu - tokeny razem: {response_without.usage.total_tokens}")
print(f"Z reasoningiem - tokeny razem: {response_with.usage.total_tokens}")

## 3. Podsumowanie procesu rozumowania

Szczegóły przeprowadzonego rozumowania mozemy pobrać. 

**Ważne:** W przypadku modeli od Open AI, pełny proces rozumowania nie jest udostępniany użytkownikowi - dostępne jest tylko **podsumowanie** (summary) tego procesu.

**Reasoning tokens vs. tokeny odpowiedzi:**
- **Reasoning tokens** - tokeny używane przez model do wewnętrznego procesu myślenia (niewidoczne dla użytkownika; dostępne jest tylko podsumowanie)
- **Tokeny odpowiedzi** - tokeny tworzące finalną odpowiedź widoczną dla użytkownika
- Suma tych dwóch składa się na `output_tokens`

**Dostęp do podsumowania rozumowania:**
- Podsumowanie dostępne poprzez iterację po `response.output` i wyszukanie elementu typu `"reasoning"`
- Parametr `reasoning={"summary": "detailed"}` zapewnia bardziej szczegółowe podsumowanie procesu myślenia
- Liczba reasoning tokens: `response.usage.output_tokens_details.reasoning_tokens`

In [ ]:
# Problem wymagający wieloetapowego rozumowania
problem = """
Firma ma 100 pracowników. 60% z nich pracuje zdalnie. 
Spośród pracujących zdalnie, 40% mieszka w Warszawie.
Spośród pracujących stacjonarnie, 75% mieszka w Warszawie.
Ile osób pracujących w tej firmie mieszka w Warszawie?
"""

response = client.responses.create(
    model="gpt-5.2",
    input=[
        {"role": "system", "content": "Jesteś pomocnym asystentem matematycznym."},
        {"role": "user", "content": problem}
    ],
    reasoning={
        "effort": "high",
        "summary": "detailed"  # Kluczowy parametr dla treści reasoningu
    },
    max_output_tokens=4000
)

# Ekstrakcja reasoning summary i odpowiedzi
for item in response.output:
    if item.type == "reasoning" and item.summary:
        print("=== PROCES ROZUMOWANIA ===")
        print(textwrap.fill(item.summary[0].text, width=80))
        print()
    elif item.type == "message":
        print("=== ODPOWIEDŹ ===")
        print(item.content[0].text)

# Lub szybko: tylko finalna odpowiedź
print("\n=== FINALNA ODPOWIEDŹ ===")
print(response.output_text)

# Statystyki reasoning tokens
print("\n=== STATYSTYKI ===")
print(f"Reasoning tokens: {response.usage.output_tokens_details.reasoning_tokens}")
print(f"Output tokens:    {response.usage.output_tokens}")
print(f"Total tokens:     {response.usage.total_tokens}")

## 4. Wpływ poziomu reasoningu na jakość odpowiedzi

Przetestujmy jak różne poziomy `reasoning_effort` wpływają na jakość odpowiedzi dla tego samego złożonego problemu.

In [ ]:
# Trudny problem logiczny
problem = """
Przykłady:
1 2 3 : 7
4 1 1 : 28
2 2 2 : 14
3 2 1 : 21
1 1 1 : 4
2 1 1 : 10

Zadanie testowe:
3 3 3 : ?

Uzupełnij brakujące wartości w zadaniu testowym powyżej.

Format odpowiedzi:
Reguła: {krótki czystotekstowy wzór wyliczenia d, bez latexa ani formatowania}
Uzupełniony ciąg z zadania : {format: a b c : d}
"""

# Testujemy różne poziomy reasoningu
reasoning_levels = ["none", "low", "medium", "high", "xhigh"]

for level in reasoning_levels:
    print(f"\n{'='*60}")
    print(f"REASONING EFFORT: {level.upper()}")
    print(f"{'='*60}\n")
    
    # Budowanie konfiguracji dla Response API
    response = client.responses.create(
        model="gpt-5.2",
        input=[
            {"role": "system", "content": "Jesteś ekspertem od logicznych zagadek."},
            {"role": "user", "content": problem}
        ],
        reasoning={
            "effort": level,        # testowanie różnych poziomów reasoningu
            "summary": "detailed"   # szczegółowe podsumowanie procesu rozumowania
        },
        max_output_tokens=4000
    )
    
    # Wyświetlenie podsumowania rozumowania (dostępne tylko gdy level != "none")
    for item in response.output:
        if item.type == "reasoning" and item.summary:
            print("--- PODSUMOWANIE ROZUMOWANIA ---")
            print(textwrap.fill(item.summary[0].text, width=60))
            print()
    
    print(response.output_text)
    
    # Szczegółowe statystyki tokenów
    print(f"\n--- STATYSTYKI ---")
    print(f"Input tokens:     {response.usage.input_tokens}")
    print(f"Output tokens:    {response.usage.output_tokens}")
    
    # Reasoning tokens dostępne tylko gdy używamy reasoning
    if level != "none":
        print(f"Reasoning tokens: {response.usage.output_tokens_details.reasoning_tokens}")
        print(f"Answer tokens:    {response.usage.output_tokens - response.usage.output_tokens_details.reasoning_tokens}")
        print(f"% reasoningu:     {response.usage.output_tokens_details.reasoning_tokens / response.usage.output_tokens * 100:.1f}%")
    
    print(f"Total tokens:     {response.usage.total_tokens}")

## 5. Wieloturowa wymiana wiadomości z przechowywaniem po stronie serwera

Najprostszy sposób prowadzenia wieloturowej rozmowy z Responses API to przekazanie `previous_response_id` — identyfikatora poprzedniej odpowiedzi. Serwer OpenAI automatycznie rekonstruuje pełną historię rozmowy (wiadomości i tokeny rozumowania), dzięki czemu:
- Klient wysyła tylko **nową wiadomość użytkownika** — nie trzeba zarządzać kontekstem ręcznie
- Tokeny rozumowania z poprzednich tur są automatycznie dostępne dla modelu
- Kod jest znacznie prostszy niż przy podejściu bezstanowym

**Kiedy to podejście nie wystarczy:** Organizacje z polityką Zero Data Retention (ZDR) nie mogą korzystać z przechowywania stanu po stronie serwera. W takim przypadku należy użyć podejścia z zaszyfrowanymi tokenami (`store=False`) pokazanego w sekcji 6.

**Mechanizm działania:**
1. Tura 1: zwykłe wywołanie, odpowiedź zostaje zapisana na serwerze pod `response.id`
2. Kolejne tury: `previous_response_id=response_N.id` + tylko nowa wiadomość w `input`
3. Serwer uzupełnia brakującą historię — model widzi pełny kontekst rozmowy

> **⚠️ Uwaga — cel dydaktyczny:** W poniższym przykładzie LLM jest wykorzystywany do szeregowania zadań wyłącznie w celu demonstracji wieloturowego reasoningu. W zastosowaniach produkcyjnych tego rodzaju problemy (szeregowanie zadań, ścieżka krytyczna, optymalizacja harmonogramów) należy rozwiązywać dedykowanymi metodami: algorytmami kombinatorycznymi, programowaniem liniowym całkowitoliczbowym (ILP) lub narzędziami takimi jak [Google OR-Tools](https://developers.google.com/optimization/scheduling). LLM nie gwarantuje poprawności ani optymalności rozwiązania.

In [ ]:
from openai import OpenAI
import os

api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

In [ ]:
# === TURA 1 ===
print("=== TURA 1 ===")
user_message_1 = """
Masz 5 zadań do wykonania na 3 maszynach (każde zadanie może trafić na dowolną maszynę):
  Zadanie A: 3h
  Zadanie B: 5h
  Zadanie C: 2h
  Zadanie D: 1h
  Zadanie E: 4h

Zależności (poprzednik musi się zakończyć, zanim następnik może ruszyć):
  A → C, B → C, B → D, C → E

Wyznacz ścieżkę krytyczną i minimalny czas ukończenia wszystkich zadań.
Podaj pełny harmonogram: które zadanie, na której maszynie, w jakich godzinach.
"""
print(f"Użytkownik: {user_message_1}\n")

response_1 = client.responses.create(
    model="gpt-5.2",
    input=[{"role": "user", "content": user_message_1}],
    reasoning={"effort": "low"},
    max_output_tokens=10000,
    # store=True jest domyślne – odpowiedź zostaje zapisana na serwerze pod response_1.id
)

In [ ]:
print(f"Model: {response_1.output_text}")

In [ ]:
print(f"ID odpowiedzi:      {response_1.id}")
print(f"Tokeny wejściowe:   {response_1.usage.input_tokens}")
print(f"Tokeny wyjściowe:   {response_1.usage.output_tokens}")
print(f"Tokeny rozumowania: {response_1.usage.output_tokens_details.reasoning_tokens}")

In [ ]:
# === TURA 2 ===
print("=== TURA 2 ===")
user_message_2 = """
W godzinie 4 jedna z maszyn ulega awarii i wypada z użytku do końca harmonogramu.
Zostają tylko 2 sprawne maszyny.

Czy harmonogram z tury 1 jest nadal wykonalny? Jeśli nie, zaproponuj nowy
harmonogram i podaj nowy minimalny czas ukończenia wszystkich zadań.
"""
print(f"Użytkownik: {user_message_2}\n")

response_2 = client.responses.create(
    model="gpt-5.2",
    input=[{"role": "user", "content": user_message_2}],
    previous_response_id=response_1.id,  # serwer rekonstruuje historię z tury 1
    reasoning={"effort": "low"},
    max_output_tokens=10000,
)

In [ ]:
print(f"Model: {response_2.output_text}")

In [ ]:
print(f"ID odpowiedzi:      {response_2.id}")
print(f"Tokeny wejściowe:   {response_2.usage.input_tokens}")
print(f"Tokeny wyjściowe:   {response_2.usage.output_tokens}")
print(f"Tokeny rozumowania: {response_2.usage.output_tokens_details.reasoning_tokens}")

In [ ]:
# === TURA 3 ===
print("=== TURA 3 ===")
user_message_3 = "A jakby to wyglądało, gdyby od początku były dostępne tylko 2 maszyny?"
print(f"Użytkownik: {user_message_3}\n")

response_3 = client.responses.create(
    model="gpt-5.2",
    input=[{"role": "user", "content": user_message_3}],
    previous_response_id=response_2.id,  # serwer rekonstruuje historię z tur 1 i 2
    reasoning={"effort": "low"},
    max_output_tokens=10000,
)

In [ ]:
print(f"Model: {response_3.output_text}")

In [ ]:
print(f"ID odpowiedzi:      {response_3.id}")
print(f"Tokeny wejściowe:   {response_3.usage.input_tokens}")
print(f"Tokeny wyjściowe:   {response_3.usage.output_tokens}")
print(f"Tokeny rozumowania: {response_3.usage.output_tokens_details.reasoning_tokens}")

In [ ]:
# Podsumowanie łącznego zużycia tokenów przez całą rozmowę
total_input = (response_1.usage.input_tokens + response_2.usage.input_tokens
               + response_3.usage.input_tokens)
total_output = (response_1.usage.output_tokens + response_2.usage.output_tokens
                + response_3.usage.output_tokens)
total_reasoning = (response_1.usage.output_tokens_details.reasoning_tokens
                   + response_2.usage.output_tokens_details.reasoning_tokens
                   + response_3.usage.output_tokens_details.reasoning_tokens)

print(f"\n=== PODSUMOWANIE CAŁEJ ROZMOWY ===")
print(f"Łączne tokeny wejściowe:   {total_input}")
print(f"Łączne tokeny wyjściowe:   {total_output}")
print(f"Łączne tokeny rozumowania: {total_reasoning}")

## 6. Wieloturowa wymiana wiadomości z zaszyfrowanymi tokenami rozumowania (ZDR)

Sekcja 5 pokazała prostsze podejście z `previous_response_id`. Gdy jednak organizacja stosuje politykę **Zero Data Retention (ZDR)** i nie może przechowywać danych na serwerach OpenAI, konieczne jest podejście bezstanowe (`store=False`) z **zaszyfrowanymi tokenami rozumowania** (`reasoning.encrypted_content`) rozwiązują ten problem w trybie bezstanowym (`store=False`):
- Serwer OpenAI **nie przechowuje** historii rozmowy — wymagane np. przez organizacje z polityką Zero Data Retention (ZDR)
- Zaszyfrowane tokeny są dołączane do odpowiedzi i przekazywane z powrotem przez klienta w kolejnej turze
- Model **odszyfrowuje je wyłącznie w pamięci operacyjnej** podczas generowania odpowiedzi, po czym są bezpiecznie usuwane
- Dzięki temu spójność łańcucha rozumowania jest zachowana mimo braku stanu po stronie serwera

**Mechanizm działania:**
1. Wywołanie z `include=["reasoning.encrypted_content"]` → odpowiedź zawiera zaszyfrowane tokeny rozumowania
2. `context += response.output` dołącza do kontekstu zarówno tekst odpowiedzi, jak i zaszyfrowane tokeny
3. Kolejne wywołanie z tym samym kontekstem pozwala modelowi kontynuować swój łańcuch rozumowania

> **⚠️ Uwaga — cel dydaktyczny:** W poniższym przykładzie LLM jest wykorzystywany do szeregowania zadań wyłącznie w celu demonstracji wieloturowego reasoningu. W zastosowaniach produkcyjnych tego rodzaju problemy (szeregowanie zadań, ścieżka krytyczna, optymalizacja harmonogramów) należy rozwiązywać dedykowanymi metodami: algorytmami kombinatorycznymi, programowaniem liniowym całkowitoliczbowym (ILP) lub narzędziami takimi jak [Google OR-Tools](https://developers.google.com/optimization/scheduling). LLM nie gwarantuje poprawności ani optymalności rozwiązania.

In [ ]:
from openai import OpenAI
import os
import json

api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def _truncate(obj, max_len=80):
    """Rekurencyjnie skraca długie ciągi znaków dla czytelności wydruku."""
    if isinstance(obj, str):
        if len(obj) <= max_len:
            return obj
        return obj[:max_len] + f'...[+{len(obj) - max_len} znaków]'
    if isinstance(obj, dict):
        return {k: _truncate(v, max_len) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_truncate(i, max_len) for i in obj]
    return obj

def print_context(ctx):
    """Wypisuje kontekst rozmowy jako sformatowany JSON.
    Zaszyfrowane tokeny rozumowania są skracane dla czytelności.
    """
    serializable = []
    for item in ctx:
        d = item if isinstance(item, dict) else item.model_dump()  # Pydantic -> dict
        serializable.append(_truncate(d))
    print(json.dumps(serializable, indent=2, ensure_ascii=False))

In [ ]:
# Inicjalizacja kontekstu wieloturowej rozmowy
# Kontekst będzie rozrastał się z każdą turą – przechowuje wiadomości oraz zaszyfrowane tokeny
context = []

In [ ]:
# === TURA 1 ===
print("=== TURA 1 ===")
user_message_1 = """
Masz 5 zadań do wykonania na 3 maszynach (każde zadanie może trafić na dowolną maszynę):
  Zadanie A: 3h
  Zadanie B: 5h
  Zadanie C: 2h
  Zadanie D: 1h
  Zadanie E: 4h

Zależności (poprzednik musi się zakończyć, zanim następnik może ruszyć):
  A → C, B → C, B → D, C → E

Wyznacz ścieżkę krytyczną i minimalny czas ukończenia wszystkich zadań.
Podaj pełny harmonogram: które zadanie, na której maszynie, w jakich godzinach.
"""
context.append({"role": "user", "content": user_message_1})
print(f"Użytkownik: {user_message_1}\n")

response_1 = client.responses.create(
    model="gpt-5.2",
    input=context,
    reasoning={"effort": "low"},
    max_output_tokens=10000,
    store=False,
    include=["reasoning.encrypted_content"]
)

context += response_1.output

In [ ]:
print(f"Model: {response_1.output_text}")

In [ ]:
print(f"Tokeny wejściowe:   {response_1.usage.input_tokens}")
print(f"Tokeny wyjściowe:   {response_1.usage.output_tokens}")
print(f"Tokeny rozumowania: {response_1.usage.output_tokens_details.reasoning_tokens}")

In [ ]:
print("\n--- Kontekst po turze 1 ---")
print_context(context)

In [ ]:
# === TURA 2 ===
print("=== TURA 2 ===")
user_message_2 = """
W godzinie 4 jedna z maszyn ulega awarii i wypada z użytku do końca harmonogramu.
Zostają tylko 2 sprawne maszyny.

Czy harmonogram z tury 1 jest nadal wykonalny? Jeśli nie, zaproponuj nowy
harmonogram i podaj nowy minimalny czas ukończenia wszystkich zadań.
"""
context.append({"role": "user", "content": user_message_2})
print(f"Użytkownik: {user_message_2}\n")

response_2 = client.responses.create(
    model="gpt-5.2",
    input=context,
    reasoning={"effort": "low"},
    max_output_tokens=10000,
    store=False,
    include=["reasoning.encrypted_content"]
)

context += response_2.output

In [ ]:
print(f"Model: {response_2.output_text}")

In [ ]:
print(f"Tokeny wejściowe:   {response_2.usage.input_tokens}")
print(f"Tokeny wyjściowe:   {response_2.usage.output_tokens}")
print(f"Tokeny rozumowania: {response_2.usage.output_tokens_details.reasoning_tokens}")

In [ ]:
print("\n--- Kontekst po turze 2 ---")
print_context(context)

In [ ]:
# === TURA 3 ===
print("=== TURA 3 ===")
user_message_3 = "A jakby to wyglądało, gdyby od początku były dostępne tylko 2 maszyny?"
context.append({"role": "user", "content": user_message_3})
print(f"Użytkownik: {user_message_3}\n")

response_3 = client.responses.create(
    model="gpt-5.2",
    input=context,                       # kontekst zawiera zaszyfrowane tokeny rozumowania z tur 1 i 2
    reasoning={"effort": "low"},
    max_output_tokens=10000,
    store=False,
    include=["reasoning.encrypted_content"]
)

context += response_3.output

In [ ]:
print(f"Model: {response_3.output_text}")

In [ ]:
print(f"Tokeny wejściowe:   {response_3.usage.input_tokens}")
print(f"Tokeny wyjściowe:   {response_3.usage.output_tokens}")
print(f"Tokeny rozumowania: {response_3.usage.output_tokens_details.reasoning_tokens}")

In [ ]:
print("\n--- Kontekst po turze 3 ---")
print_context(context)

In [ ]:
# Podsumowanie łącznego zużycia tokenów przez całą rozmowę
total_input = (response_1.usage.input_tokens + response_2.usage.input_tokens
               + response_3.usage.input_tokens)
total_output = (response_1.usage.output_tokens + response_2.usage.output_tokens
                + response_3.usage.output_tokens)
total_reasoning = (response_1.usage.output_tokens_details.reasoning_tokens
                   + response_2.usage.output_tokens_details.reasoning_tokens
                   + response_3.usage.output_tokens_details.reasoning_tokens)

print(f"\n=== PODSUMOWANIE CAŁEJ ROZMOWY ===")
print(f"Łączne tokeny wejściowe:   {total_input}")
print(f"Łączne tokeny wyjściowe:   {total_output}")
print(f"Łączne tokeny rozumowania: {total_reasoning}")

**Reasoning warto stosować gdy:**
- Problem wymaga wieloetapowej analizy
- Istnieje ryzyko popełnienia błędu w rozumowaniu
- Potrzebna jest wysoka dokładność odpowiedzi
- Zadanie wymaga weryfikacji hipotez
- Pracujemy z matematyką, logiką lub złożonymi algorytmami

**Reasoning może być zbędny gdy:**
- Zadanie jest bardzo proste
- Potrzebna jest kreatywna, swobodna odpowiedź
- Zależy nam na szybkości, a nie perfekcyjnej dokładności
- Koszt dodatkowych tokenów jest istotny

**Porównanie poziomów reasoningu:**
- `reasoning_effort="none"` - brak reasoningu, najszybsza odpowiedź (domyślne)
- `reasoning_effort="low"` - minimalne dodatkowe koszty i czas
- `reasoning_effort="medium"` - umiarkowane koszty, zbalansowane podejście
- `reasoning_effort="high"` - znaczące koszty, bardzo dobra jakość
- `reasoning_effort="xhigh"` - maksymalne koszty i czas (nawet 5-10 minut), najwyższa jakość dla krytycznych zadań

**Pamiętaj:** Modele reasoningowe nie obsługują parametrów `temperature`, `top_p`, `presence_penalty`, `frequency_penalty` oraz innych parametrów kontrolujących próbkowanie - zamiast nich używamy `reasoning_effort` do kontroli głębokości analizy.